# Reto 1: Consulta y análisis de datos meteorológicos

## 10. Controla errores y datos ausentes

Un programa debe responder de forma comprensible cuando falla la conexión, la API tarda demasiado o el servidor devuelve un error. También debe comprobar que los datos recibidos son válidos antes de calcular estadísticas.

## Petición segura a la API

La función siguiente controla por separado los errores más habituales de una petición. Devuelve los datos cuando todo funciona y `None` cuando se produce un error, evitando que el programa continúe con información no válida.

In [1]:
import requests

url = "https://archive-api.open-meteo.com/v1/archive"
parametros = {
    "latitude": 37.7796,
    "longitude": -3.7849,
    "start_date": "2025-01-01",
    "end_date": "2025-01-31",
    "daily": "temperature_2m_mean,temperature_2m_max,temperature_2m_min",
    "timezone": "Europe/Madrid",
}

def obtener_datos(url, parametros):
    try:
        respuesta = requests.get(url, params=parametros, timeout=30)
        respuesta.raise_for_status()
        return respuesta.json()
    except requests.exceptions.Timeout:
        print("Error: la API ha tardado demasiado en responder.")
    except requests.exceptions.ConnectionError:
        print("Error: no se ha podido establecer la conexión con la API.")
    except requests.exceptions.HTTPError as error:
        print("Error HTTP devuelto por el servidor:", error)
    except requests.exceptions.JSONDecodeError:
        print("Error: la respuesta recibida no contiene un JSON válido.")
    except requests.exceptions.RequestException as error:
        print("Error inesperado durante la petición:", error)

    return None

## Validación de los datos recibidos

Antes de realizar cálculos se comprueba que:

- La respuesta sea un diccionario y contenga la clave `daily`.
- `daily` incluya las fechas y las tres series de temperaturas solicitadas.
- Todos esos elementos sean listas y no estén vacíos.
- Las listas tengan la misma longitud.
- Ninguna lista contenga valores ausentes (`None`).

In [2]:
def validar_datos(datos):
    if not isinstance(datos, dict):
        return False, "La respuesta no es un diccionario."

    if "daily" not in datos or not isinstance(datos["daily"], dict):
        return False, "La respuesta no contiene un diccionario 'daily'."

    diarios = datos["daily"]
    claves_necesarias = [
        "time",
        "temperature_2m_mean",
        "temperature_2m_max",
        "temperature_2m_min",
    ]

    claves_ausentes = [clave for clave in claves_necesarias if clave not in diarios]
    if claves_ausentes:
        return False, f"Faltan las claves: {', '.join(claves_ausentes)}."

    if any(not isinstance(diarios[clave], list) for clave in claves_necesarias):
        return False, "Las fechas y temperaturas deben estar almacenadas en listas."

    if any(len(diarios[clave]) == 0 for clave in claves_necesarias):
        return False, "Una o varias listas están vacías."

    longitudes = {clave: len(diarios[clave]) for clave in claves_necesarias}
    if len(set(longitudes.values())) != 1:
        return False, f"Las listas no tienen la misma longitud: {longitudes}."

    for clave in claves_necesarias:
        if any(valor is None for valor in diarios[clave]):
            return False, f"La lista '{clave}' contiene valores ausentes."

    return True, "Los datos son válidos."

## Programa principal

Solo se calculan estadísticas si la petición termina correctamente y la validación confirma que los datos son válidos.

In [3]:
datos = obtener_datos(url, parametros)

if datos is not None:
    datos_validos, mensaje = validar_datos(datos)
    print(mensaje)

    if datos_validos:
        temperaturas = datos["daily"]["temperature_2m_mean"]
        media = sum(temperaturas) / len(temperaturas)
        print(f"Temperatura media del periodo: {media:.2f} °C")

Los datos son válidos.
Temperatura media del periodo: 8.82 °C


## Pruebas con datos ausentes e incoherentes

Estas pruebas locales permiten comprobar la validación sin tener que provocar un fallo real en la API.

In [4]:
datos_con_valor_ausente = {
    "daily": {
        "time": ["2025-01-01", "2025-01-02"],
        "temperature_2m_mean": [7.1, None],
        "temperature_2m_max": [12.8, 12.9],
        "temperature_2m_min": [2.4, 2.5],
    }
}

datos_con_longitudes_distintas = {
    "daily": {
        "time": ["2025-01-01", "2025-01-02"],
        "temperature_2m_mean": [7.1],
        "temperature_2m_max": [12.8, 12.9],
        "temperature_2m_min": [2.4, 2.5],
    }
}

print(validar_datos(datos_con_valor_ausente))
print(validar_datos(datos_con_longitudes_distintas))

(False, "La lista 'temperature_2m_mean' contiene valores ausentes.")
(False, "Las listas no tienen la misma longitud: {'time': 2, 'temperature_2m_mean': 1, 'temperature_2m_max': 2, 'temperature_2m_min': 2}.")


## Comprueba tu comprensión

### 1. ¿Qué errores se controlan durante la petición?

**Respuesta:**

- `Timeout`: la API tarda más de 30 segundos en responder.
- `ConnectionError`: no puede establecerse la conexión.
- `HTTPError`: el servidor devuelve un código de error, como `404` o `500`.
- `JSONDecodeError`: la respuesta no contiene un JSON válido.
- `RequestException`: controla otros errores relacionados con la petición que no pertenecen a los casos anteriores.

### 2. ¿Por qué se comprueba la clave `daily`?

**Respuesta:**

Las fechas y temperaturas se encuentran dentro de `datos["daily"]`. Si la clave no existe y se intenta acceder directamente a ella, Python genera un `KeyError`. Además, su ausencia puede indicar que la API ha devuelto una estructura distinta de la esperada.

### 3. ¿Por qué deben coincidir las longitudes de las listas?

**Respuesta:**

Cada posición representa el mismo día en todas las listas. Si las longitudes son diferentes, alguna fecha no tendrá todos sus valores asociados y podrían relacionarse temperaturas con días incorrectos.

### 4. ¿Por qué se buscan valores ausentes antes de calcular estadísticas?

**Respuesta:**

Un valor `None` no puede sumarse ni compararse directamente con números. Intentar calcular una media, un máximo o un mínimo sin comprobarlo produciría un `TypeError` o un resultado incompleto.

### 5. ¿Por qué no debe sustituirse automáticamente un valor ausente por cero?

**Respuesta:**

Un dato ausente significa que la temperatura es desconocida, no que fuese de `0 °C`. Sustituirlo por cero introduciría una observación falsa: reduciría artificialmente la media, podría convertirse en el mínimo del periodo y aumentaría la dispersión. Esto llevaría a conclusiones meteorológicas incorrectas.

La decisión adecuada depende del análisis: se puede excluir el registro avisando de ello, solicitar de nuevo los datos o aplicar una técnica de estimación justificada. En todos los casos debe documentarse el tratamiento realizado.